In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os
import numba
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse import issparse
import seaborn as sns
import matplotlib.lines as mlines
import rapids_singlecell as rsc

sc.set_figure_params(figsize=(3,3),dpi=150)

In [ ]:
adata_anno = sc.read_h5ad('adata_anno_score_genes_rank_re.h5ad')
adata_anno

In [ ]:
adata_anno.obs

In [ ]:
adata_bcell = sc.read_h5ad('leiden_detailed/adata_b.h5ad')
adata_endo = sc.read_h5ad('leiden_detailed/adata_endo.h5ad')
adata_mast = sc.read_h5ad('leiden_detailed/adata_mast.h5ad')
adata_mye = sc.read_h5ad('leiden_detailed/adata_mye.h5ad')
adata_nk = sc.read_h5ad('leiden_detailed/adata_nk.h5ad')
adata_stromal = sc.read_h5ad('leiden_detailed/adata_s.h5ad')
adata_tcell = sc.read_h5ad('leiden_detailed/adata_t.h5ad')
adata_epi = sc.read_h5ad('leiden_detailed/adata_epi.h5ad')

In [ ]:
print(adata_bcell)
print(adata_endo)
print(adata_mast)
print(adata_mye)
print(adata_nk)
print(adata_stromal)
print(adata_tcell)
print(adata_epi)


In [ ]:
adata_dict = {
    'bcell': adata_bcell,
    'endo': adata_endo,
    'mast': adata_mast,
    'mye': adata_mye,
    'nk': adata_nk,
    'stromal': adata_stromal,
    'tcell': adata_tcell,
    'epi':adata_epi
}

for name, adata in adata_dict.items():
    sc.pl.umap(
        adata, 
        color='cell_subtype', 
        title=f'{name} - cell_subtype', 
        show=True
    )


In [ ]:
def get_index_to_subtype_dict(adata):
    return adata.obs['cell_subtype'].to_dict()

bcell_dict = get_index_to_subtype_dict(adata_bcell)
endo_dict = get_index_to_subtype_dict(adata_endo)
mast_dict = get_index_to_subtype_dict(adata_mast)
mye_dict = get_index_to_subtype_dict(adata_mye)
nk_dict = get_index_to_subtype_dict(adata_nk)
stromal_dict = get_index_to_subtype_dict(adata_stromal)
tcell_dict = get_index_to_subtype_dict(adata_tcell)
epi_dict = get_index_to_subtype_dict(adata_epi)

for name, d in zip(
    ['bcell', 'endo', 'mast', 'mye', 'nk', 'stromal', 'tcell','epi'],
    [bcell_dict, endo_dict, mast_dict, mye_dict, nk_dict, stromal_dict, tcell_dict,epi_dict]
):
    print(f"\n{name} (total {len(d)} cells):")
    print(dict(list(d.items())[:5]))  # show only the first five entries to avoid overly long output

In [ ]:
merged_dict = {
    **bcell_dict,
    **endo_dict,
    **mast_dict,
    **mye_dict,
    **nk_dict,
    **stromal_dict,
    **tcell_dict,
    **epi_dict
}


print(f"Total entries in merged_dict: {len(merged_dict)}")

print(dict(list(merged_dict.items())[:5]))

In [ ]:
adata_anno.obs['cell_subtype'] = adata_anno.obs.index.map(merged_dict)

In [ ]:
sc.pl.umap(adata_anno,color='leiden_coarse')

In [ ]:

adata_anno = adata_anno[adata_anno.obs['cell_subtype'].notna(), :].copy()
adata_anno

In [ ]:
sc.pl.umap(adata_anno,color='leiden_coarse')

In [ ]:
adata_anno.obs['cell_type'] = adata_anno.obs['cell_subtype'].str.split('_').str[0]

In [ ]:
t_class_dict = {'T_FOXP3':'CD4+ T',
                'T_GNLY':'CD8+ T',
                'T_GZMH':'CD8+ T',
                'T_GZMK':'CD8+ T',
                'T_HSPA1A':'CD8+ T',
                'T_IL7R':'CD4+ T',
                'T_KLRB1':'CD4+ T',
                'T_LAG3':'CD8+ T',
                'T_MKI67':'CD8+ T',
                'T_RPs':'CD8+ T',
                'T_TOX':'CD8+ T',
                'T_TRBC1':'CD8+ T',}
adata_anno.obs['cell_type_fine'] = adata_anno.obs['cell_subtype'].map(t_class_dict).fillna(adata_anno.obs['cell_type'])

In [ ]:
sc.pl.umap(adata_anno,color=['leiden_coarse','cell_type','cell_type_fine'],wspace=0.5,save='allcells_cell_type')

In [ ]:
adata_anno.write_h5ad('adata_anno_cell_subtype_re.h5ad')

In [ ]:
# Replot all-cell-type dotplot from saved annotated h5ad.
adata_anno_replot = sc.read_h5ad('adata_anno_cell_subtype_re.h5ad')
adata_anno_replot


In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'

markers_replot_all_cell_type = {
    "B Cells": ["CD79B", "BANK1", "MS4A1"],
    "Endothelial Cells": ["VWF", "FLT1", "PECAM1"],
    "Epithelial Cells": ["KRT18", "KRT8", "EPCAM"],
    "Mast Cells": ["MS4A2", "TPSAB1", "KIT"],
    "Myeloid Cells": ["CD14", "LYZ", "CD68"],
    "NK Cells": ["NKG7", "GNLY", "KLRD1"],
    "Stromal Cells": ["RGS5", "TAGLN", "ACTA2"],
    "T Cells": ["CD3D", "CD3E", "CD3G"],
    "pDC": ["CLEC4C", "IL3RA", "LILRA4"],
}

sc.pl.dotplot(
    adata_anno_replot,
    var_names=markers_replot_all_cell_type,
    groupby='cell_type',
    standard_scale='var',
    use_raw=True,
    save='_all_cell_type.pdf',
)


In [ ]:
os.makedirs("figures", exist_ok=True)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


In [ ]:
adata = sc.read_h5ad('adata_anno_cell_subtype_re.h5ad')

In [ ]:


series_counts = adata.obs['series'].value_counts().sort_index()
total = series_counts.sum()
legend_labels = [
    f"{name} (n={count}, {count/total:.1%})"
    for name, count in zip(series_counts.index, series_counts.values)
]


fig, ax = plt.subplots(figsize=(7, 7))
patches, texts = ax.pie(
    series_counts,
    labels=None,
    autopct=None,
    #startangle=90,
    #wedgeprops=dict(edgecolor='w')
)
plt.title("", fontsize=16)


ax.legend(
    patches, legend_labels,
    title="",
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    fontsize=8,
    title_fontsize=13,frameon=False,
    handlelength=1, handleheight=1
)

plt.tight_layout()
plt.savefig('figures/single_circle.pdf',bbox_inches='tight',dpi=600)
plt.show()


In [ ]:
from matplotlib.patches import Patch
import matplotlib


sample_df = (
    adata.obs[['series', 'sample', 'status']]
    .drop_duplicates()
    .groupby(['series', 'status'])
    .size()
    .reset_index(name='sample_count')
)
series_order = sample_df['series'].drop_duplicates().tolist()
status_order = ['tumor', 'normal-like']
tab10 = matplotlib.cm.get_cmap('tab10').colors

# Inner ring: total sample count for each series
inner_sizes = [
    sample_df[sample_df['series'] == s]['sample_count'].sum()
    for s in series_order
]
total_samples = sum(inner_sizes)
inner_colors = [tab10[i % 10] for i in range(len(series_order))]

# Outer ring: tumor and normal-like sample counts for each series
outer_sizes = []
outer_colors = []
status_color_map = {'tumor': 'skyblue', 'normal-like': '#FFD580'}
for s in series_order:
    for st in status_order:
        row = sample_df[(sample_df['series'] == s) & (sample_df['status'] == st)]
        cnt = int(row['sample_count'].iloc[0]) if not row.empty else 0
        outer_sizes.append(cnt)
        outer_colors.append(status_color_map[st])

fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)


wedges1, _ = ax.pie(
    inner_sizes,
    radius=0.9,
    labels=None,
    colors=inner_colors
)


wedges2, _ = ax.pie(
    outer_sizes,
    radius=1.0,
    labels=None,
    wedgeprops=dict(width=0.1),
    colors=outer_colors
)

ax.set_aspect('equal')
plt.title("", fontsize=15)


series_handles = [Patch(facecolor=inner_colors[i], edgecolor='w') for i in range(len(series_order))]
series_legend_labels = [
    f"{s} (n={n}, {n/total_samples:.1%})"
    for s, n in zip(series_order, inner_sizes)
]

# Status legend
status_handles = [Patch(facecolor=status_color_map[st], edgecolor='w') for st in status_order]
status_legend_labels = status_order


leg1 = ax.legend(
    series_handles,
    series_legend_labels,
    title="Series",
    loc="upper left",
    bbox_to_anchor=(0.85, 1.00),
    fontsize=8,
    frameon=False,
    handlelength=1, handleheight=1,
    title_fontsize=12
)
ax.add_artist(leg1)

leg2 = ax.legend(
    status_handles,
    status_legend_labels,
    title="Status",
    loc="lower left",
    bbox_to_anchor=(1.05, 0.00),
    fontsize=9,
    frameon=False,
    handlelength=1, handleheight=1,
    title_fontsize=12
)


plt.subplots_adjust(right=1)
plt.savefig('figures/double_circle.pdf',bbox_inches='tight',dpi=600)
plt.show()


In [ ]:


cell_counts = (
    adata.obs
    .groupby(['sample', 'cell_type'])
    .size()
    .reset_index(name='count')
)


total_counts = (
    adata.obs.groupby('sample').size().reset_index(name='total_count')
)


cell_counts = cell_counts.merge(total_counts, on='sample')
cell_counts['proportion'] = cell_counts['count'] / cell_counts['total_count']

cell_type_order = list(adata.uns['cell_type_colors'].keys()) \
    if isinstance(adata.uns['cell_type_colors'], dict) \
    else adata.obs['cell_type'].cat.categories.tolist()
palette = adata.uns['cell_type_colors']


plt.figure(figsize=(5, 5))

sns.boxplot(
    data=cell_counts,
    x='cell_type',
    y='proportion',
    palette=palette,
    order=cell_type_order,
    showfliers=False,
    width=0.6
)



plt.ylabel('Proportion in each sample')
plt.xlabel('Cell type')
plt.title('Cell type composition per sample')
plt.xticks(rotation=90)
plt.ylim(-0.05, 1.05)
plt.yticks([0, 0.25, 0.5, 0.75, 1.0])
plt.tight_layout()
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.savefig('figures/boxplot.pdf',bbox_inches='tight')
plt.show()
